# 03 - One-Class SVM Baseline

Trains a One-Class SVM on a 50,000-row stratified sample of CIC-IDS2017 (full dataset computationally infeasible for this algorithm).

**Approach:** unsupervised, distance-based boundary around normal data. Feature scaling applied (matters significantly here, unlike Isolation Forest, since the algorithm relies on distance calculations).

**Key result:** 37% precision / 44% recall on the ANOMALY class — underperforms Isolation Forest on both accuracy and training time (37 sec on 2% of the data vs. Isolation Forest's 9 sec on the full dataset).

### Load data and take a representative sample

In [1]:
import pandas as pd
df = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017_cleaned.csv')
df_sample = df.sample(n=50000, random_state=42)
print(df_sample['Binary_Label'].value_counts())

Binary_Label
0    41724
1     8276
Name: count, dtype: int64


### Split features/labels and train/test

In [2]:
X = df_sample.drop(columns=['Label', 'Binary_Label'])
y = df_sample['Binary_Label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Scale the features

In [3]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train One-Class SVM

In [4]:
from sklearn.svm import OneClassSVM
import time

start = time.time()
oc_svm = OneClassSVM(kernel='rbf', nu=0.2, gamma='scale')
oc_svm.fit(X_train_scaled)
print(f'Training took {time.time() - start:.1f} seconds')

Training took 37.0 seconds


## Predict and evaluate

In [5]:
preds = oc_svm.predict(X_test_scaled)
preds_binary = [1 if p == -1 else 0 for p in preds]

from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, preds_binary))
print(classification_report(y_test, preds_binary))

[[10689  1828]
 [ 1396  1087]]
              precision    recall  f1-score   support

           0       0.88      0.85      0.87     12517
           1       0.37      0.44      0.40      2483

    accuracy                           0.79     15000
   macro avg       0.63      0.65      0.64     15000
weighted avg       0.80      0.79      0.79     15000

